In [1]:
import numpy as np
import cartopy.crs as ccrs
import easygems.healpix as egh
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr
import intake
mpl.rcParams['figure.dpi'] = 72

In [2]:
catalog = "https://digital-earths-global-hackathon.github.io/catalog/catalog.yaml"
# catalog = "/global/homes/f/feng045/program/hackathon/catalog/NERSC/main.yaml"

# current_location = "online"
current_location = "NERSC"

cat = intake.open_catalog(catalog)[current_location]
list (cat)

['CERES_EBAF',
 'ERA5',
 'IR_IMERG',
 'JRA3Q',
 'MERRA2',
 'arp-gem-1p3km',
 'arp-gem-2p6km',
 'casesm2_10km_nocumulus',
 'icon_d3hp003',
 'icon_d3hp003aug',
 'icon_d3hp003feb',
 'icon_ngc4008',
 'ifs_tco3999-ng5_deepoff',
 'ifs_tco3999-ng5_rcbmf',
 'ifs_tco3999-ng5_rcbmf_cf',
 'ifs_tco3999_rcbmf',
 'nicam_220m_test',
 'nicam_gl11',
 'scream-dkrz',
 'scream2D_hrly',
 'scream_lnd',
 'scream_ne120',
 'scream_ne120_inst',
 'tracking',
 'tracking-d3hp003',
 'um_Africa_km4p4_RAL3P3_n1280_GAL9_nest',
 'um_CTC_km4p4_RAL3P3_n1280_GAL9_nest',
 'um_SAmer_km4p4_RAL3P3_n1280_GAL9_nest',
 'um_SEA_km4p4_RAL3P3_n1280_GAL9_nest',
 'um_glm_n1280_CoMA9_TBv1p2',
 'um_glm_n1280_GAL9',
 'um_glm_n2560_RAL3p3',
 'wrf_conus',
 'wrf_samerica']

In [3]:
source = "icon_d3hp003"
pd.DataFrame(cat[source].describe()["user_parameters"])

,name,description,type,allowed,default
0,time,time resolution of the dataset,str,"[PT1H, PT3H, PT6H, P1D]",P1D
1,time_method,time subsetting method,str,"[mean, inst]",mean
2,zoom,zoom resolution of the dataset,int,"[11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1, 0]",0


In [4]:
catalog_params = {
    "zoom": 8,
    # "time": "PT1H",
    # "time": "PT3H",
    "time": "PT6H",
    "time_method": "inst",
}

# Read PT6H 3D data
ds3d = cat[source](**catalog_params).to_dask()
ds3d = ds3d.pipe(egh.attach_coords)
ds3d

/global/common/software/m1867/python/hackathon/lib/python3.12/site-packages/intake_xarray/base.py:21: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  'dims': dict(self._ds.dims),


<xarray.Dataset> Size: 1TB
Dimensions:       (time: 1700, cell: 786432, pressure: 30, pressure_rva: 3)
Coordinates:
  * pressure      (pressure) int64 240B 5 10 20 50 ... 92500 95000 97500 100000
  * pressure_rva  (pressure_rva) int64 24B 16 18 23
  * time          (time) datetime64[ns] 14kB 2020-01-01T06:00:00 ... 2021-03-01
    crs           int64 8B 0
  * cell          (cell) int64 6MB 0 1 2 3 4 ... 786428 786429 786430 786431
    lat           (cell) float64 6MB 0.1492 0.2984 0.2984 ... -0.2984 -0.1492
    lon           (cell) float64 6MB 45.0 45.18 44.82 45.0 ... 315.2 314.8 315.0
Data variables: (12/23)
    egpvi         (time, cell) float32 5GB ...
    einvi         (time, cell) float32 5GB ...
    ekhvi         (time, cell) float32 5GB ...
    ekvvi         (time, cell) float32 5GB ...
    hur           (time, pressure, cell) float32 160GB ...
    hus           (time, pressure, cell) float32 160GB ...
    ...            ...
    ua            (time, pressure, cell) float32 160GB ...
    uas           (time, cell) float32 5GB ...
    va            (time, pressure, cell) float32 160GB ...
    vas           (time, cell) float32 5GB ...
    wa            (time, pressure, cell) float32 160GB ...
    zg            (time, pressure, cell) float32 160GB ...

In [5]:
ds3d.wa.attrs

{'grid_mapping': 'crs',
 'hiopy::enable': True,
 'hiopy::nnn': 4,
 'hiopy::src_name': 'pl::wa_phy',
 'hiopy::time_method': 'point',
 'long_name': 'vertical velocity in m/s',
 'short_name': '',
 'standard_name': 'upward_air_velocity',
 'units': 'm s-1'}

In [6]:
ds3d.zg.attrs

{'grid_mapping': 'crs',
 'hiopy::enable': True,
 'hiopy::nnn': 4,
 'hiopy::src_name': 'pl::z_mc',
 'hiopy::time_method': 'point',
 'long_name': 'geometric height at full level center',
 'short_name': '',
 'standard_name': 'geometric_height_at_full_level_center',
 'units': 'm'}

In [15]:
# Read PT3H 2D data
ds2d = cat[source](zoom=8, time="PT3H", time_method="mean").to_dask()
ds2d = ds2d.pipe(egh.attach_coords)
ds2d

/global/common/software/m1867/python/hackathon/lib/python3.12/site-packages/intake_xarray/base.py:21: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  'dims': dict(self._ds.dims),


<xarray.Dataset> Size: 492GB
Dimensions:        (time: 3400, cell: 786432, soil_level: 5)
Coordinates:
  * soil_level     (soil_level) int64 40B 0 0 0 2 6
  * time           (time) datetime64[ns] 27kB 2020-01-01T03:00:00 ... 2021-03-01
    crs            int64 8B 0
  * cell           (cell) int64 6MB 0 1 2 3 4 ... 786428 786429 786430 786431
    lat            (cell) float64 6MB 0.1492 0.2984 0.2984 ... -0.2984 -0.1492
    lon            (cell) float64 6MB 45.0 45.18 44.82 ... 315.2 314.8 315.0
Data variables: (12/45)
    clivi          (time, cell) float32 11GB ...
    clt            (time, cell) float32 11GB ...
    clwvi          (time, cell) float32 11GB ...
    hflsd          (time, cell) float32 11GB ...
    hfssd          (time, cell) float32 11GB ...
    huss           (time, cell) float32 11GB ...
    ...             ...
    tend_ekhdynvi  (time, cell) float32 11GB ...
    tend_ekhtmxvi  (time, cell) float32 11GB ...
    tend_ekvdynvi  (time, cell) float32 11GB ...
    ts             (time, cell) float32 11GB ...
    uas            (time, cell) float32 11GB ...
    vas            (time, cell) float32 11GB ...

In [17]:
# Read PT3H 2D data
ds1h = cat[source](zoom=8, time="PT1H", time_method="inst").to_dask()
ds1h = ds1h.pipe(egh.attach_coords)
ds1h

/global/common/software/m1867/python/hackathon/lib/python3.12/site-packages/intake_xarray/base.py:21: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  'dims': dict(self._ds.dims),


<xarray.Dataset> Size: 225GB
Dimensions:  (cell: 786432, time: 10200)
Coordinates:
  * time     (time) datetime64[ns] 82kB 2020-01-01T01:00:00 ... 2021-03-01
    crs      int64 8B 0
  * cell     (cell) int64 6MB 0 1 2 3 4 5 ... 786427 786428 786429 786430 786431
    lat      (cell) float64 6MB 0.1492 0.2984 0.2984 ... -0.2984 -0.2984 -0.1492
    lon      (cell) float64 6MB 45.0 45.18 44.82 45.0 ... 315.2 314.8 315.0
Data variables:
    orog     (cell) float32 3MB ...
    pr       (time, cell) float32 32GB ...
    psl      (time, cell) float32 32GB ...
    rlut     (time, cell) float32 32GB ...
    rsut     (time, cell) float32 32GB ...
    sftgif   (cell) float32 3MB ...
    sftlf    (cell) float32 3MB ...
    ts       (time, cell) float32 32GB ...
    uas      (time, cell) float32 32GB ...
    vas      (time, cell) float32 32GB ...